# SQL4Business — Deliverable 3: sistema de agentes

Reproduce todo lo que reporta `docs/deliverable3.pdf` desde el repositorio, en el hardware declarado
en el Deliverable 1 (Colab T4). La primera celda clona el repositorio en `REF` e instala las
dependencias fijadas; no hay que subir archivos ni tener cuenta en Hugging Face.

1. `Runtime > Change runtime type > T4 GPU`.
2. Correr la celda de instalación y la **prueba corta** (unos 5 minutos) para confirmar que el modelo
   responde bien a los prompts de los agentes.
3. La **demo** muestra en vivo los mensajes entre agentes (lo que se ve en el video) y guarda las trazas
   en `results/deliverable3/demo/`.
4. La **evaluación completa** corre 8 configuraciones sobre 27 preguntas (unas 2 horas). Si Colab se
   desconecta, volver a correr la celda: `--resume` salta las configuraciones ya guardadas.
5. La última celda descarga un ZIP con los resultados para subirlos al repositorio.

In [ ]:
REPO_URL = "https://github.com/aredhel-jmze/Asistente-SQL-4-Business.git"
REF = "main"  # rama, tag o commit a evaluar

import os, subprocess, sys

os.chdir("/content")  # never stay inside a folder that a later cell may delete

ROOT = "/content/Asistente-SQL-4-Business"
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, ROOT], check=True)
subprocess.run(["git", "-C", ROOT, "fetch", "--quiet", "origin"], check=True)
# A branch is always taken at its latest version on GitHub; a tag or commit is used as is.
is_branch = subprocess.run(["git", "-C", ROOT, "rev-parse", "--verify", "--quiet", f"origin/{REF}"],
                           capture_output=True).returncode == 0
# --force discards files changed by an earlier run in the same Colab session (results, tables).
subprocess.run(["git", "-C", ROOT, "checkout", "--quiet", "--force", f"origin/{REF}" if is_branch else REF], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", f"{ROOT}/requirements.txt"], check=True)
os.chdir(ROOT)
print("commit:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

## Prueba corta

Corre el sistema de agentes completo sobre 2 preguntas por conjunto. Los resultados van a
`results/deliverable3/prueba/` y no se mezclan con la evaluación completa. Si aparecen muchas
autorrevisiones del Planificador (JSON inválido), avisar antes de lanzar la evaluación completa.

In [ ]:
!python scripts/evaluate_d3.py --limit 2 --configs agentes

## Demo: los agentes conversando

Cada línea es un mensaje: quién lo envía, a quién, qué tipo de mensaje es y su contenido
(plan JSON, llamadas a herramientas, correcciones, veredictos del Verificador). Antes se muestra la
baseline sobre la misma entrada. `q09-p` es la primera paráfrasis del conjunto externo, la misma
entrada que mostró el video del Deliverable 2 (no se eligió por su resultado). Se puede usar cualquier
`--id` de `data/questions.json` o `data/paraphrases.json`, o escribir una pregunta nueva.

In [ ]:
!python scripts/run_agents.py --id q09-p

In [ ]:
!python scripts/run_agents.py --id q14-p --no-baseline

**Caso de falla (q12).** El Verificador detecta que el plan pone agosto antes que marzo y lo devuelve
al Planificador, que repite el mismo plan. La traza queda en `results/deliverable3/demo/q12.json`.

In [ ]:
!python scripts/run_agents.py --id q12 --no-baseline

## Evaluación completa y ablación

Configuraciones, todas sobre las mismas 27 preguntas y con el mismo criterio:
`baseline` (prompt directo del D1), `d2` (solución del Deliverable 2), `agentes` (sistema completo) y
`sin_<técnica>` (el sistema sin una técnica a la vez). Escribe `results/deliverable3/<config>.json`,
una traza por pregunta del sistema completo en `results/deliverable3/trazas/` y la tabla
`docs/deliverable3_results.tex`.

In [ ]:
!python scripts/evaluate_d3.py --resume

## Descargar los resultados

Descarga `deliverable3_resultados.zip` con `results/deliverable3/` y `docs/deliverable3_results.tex`.
Su contenido se sube al repositorio respetando esas carpetas.

In [ ]:
import shutil, os
staging = "/content/entrega_d3"
shutil.rmtree(staging, ignore_errors=True)
shutil.copytree("results/deliverable3", f"{staging}/results/deliverable3")
os.makedirs(f"{staging}/docs", exist_ok=True)
shutil.copy("docs/deliverable3_results.tex", f"{staging}/docs/")
shutil.make_archive("/content/deliverable3_resultados", "zip", staging)
from google.colab import files
files.download("/content/deliverable3_resultados.zip")